# Vector Poisson equation in a rectangle

$$
\mathbb{S}_{\mathbf{u}}
\begin{cases}
\Omega = [0, L_x] \times [0, L_y] \\
\textbf{f}(x,y)=-y\textbf{e}_x + x\textbf{e}_y \\
\textbf{u}_{\text{D}}(x,y=0)=x\textbf{e}_x + \textbf{e}_y \\
\textbf{u}_{\text{D}}(x,y=L_y)=\textbf{e}_x + 2\textbf{e}_y \\
\textbf{u}_{\text{D}}(x=0,y)=\textbf{0} \\
\textbf{u}_{\text{D}}(x=L_x,y)=\textbf{e}_x -\textbf{e}_y
\end{cases}
$$

In [ ]:
from dolfinx.fem import FunctionSpace
from lucifex.mesh import rectangle_mesh, mesh_boundary
from lucifex.fem import Function, Constant, cross_section_grid
from lucifex.solver import bvp, BoundaryConditions
from lucifex.plt import plot_colormap, plot_line
from lucifex.utils.fenicsx_utils import extract_component_functions
from lucifex.pde.poisson import poisson


Lx = 2.0
Ly = 1.0
mesh = rectangle_mesh(Lx, Ly, 20, 10)
boundary = mesh_boundary(
    mesh, 
    {
        "left": lambda x: x[0],
        "right": lambda x: x[0] - Lx,
        "lower": lambda x: x[1],
        "upper": lambda x: x[1] - Ly,
    },
)
f = Function((mesh, 'P', 1, 2), lambda x: (-x[1], x[0]), name='f')

bcs = BoundaryConditions(
    ('dirichlet', boundary['left'], (0.0, 0.0)),
    ('dirichlet', boundary['right'], Constant(mesh, (1.0, -1.0))),
    ('dirichlet', boundary['lower'], (lambda x: x[0], 1.0), 1),
    ('dirichlet', boundary['upper'], 1.0, 0),
    ('dirichlet', boundary['upper'], 2.0, 1),
)

u = Function(f.function_space, name='u')
u_solver = bvp(poisson, bcs)(u, f)
u_solver.solve()

ux, uy = extract_component_functions(('P', 1), u, names=('ux', 'uy'))

RuntimeError: Interpolation data has the wrong shape/size.

In [ ]:
uxx, y_value = cross_section_grid(ux, 'y', 0.5)
plot_line(uxx, x_label='$x$', y_label=f'{u.name}_x(y={y_value:.2f})')

uxy, x_value = cross_section_grid(ux, 'x', 0.5)
plot_line(uxy, x_label='$y$', y_label=f'{u.name}_x(x={y_value:.2f})')

NameError: name 'cross_section_grid' is not defined

In [ ]:
uyy, y_value = cross_section_grid(uy, 'y', 0.5)
plot_line(uyy, x_label='$x$', y_label=f'{u.name}_y(y={y_value:.2f})')

uyx = cross_section_grid(uy, 'x', 0.5)
plot_line(uyx, x_label='$y$', y_label=f'{u.name}_y(x={y_value:.2f})')

NameError: name 'cross_section_grid' is not defined